# M20 — SRQ-Adaptive budget sensitivity and selection criterion (train-only, T4)

Three CIFAR-100 **validation** streams with the RanPAC random-ReLU head at width 20,000.
It measures (1) how accuracy and memory change with the FP16 budget under the locked
factor-MSE criterion, (2) whether a row-weighted system-error criterion lowers the
classifier error at the same budget, and (3) how much faster batched block scoring is,
with identical decisions. **No `test.pt` is ever created or read.** The contract is frozen
in `docs/research/SRQ_GENERALIZATION_M20_PROTOCOL.md`.

Select a **T4 GPU**, upload `srq_generalization_m6_width_sweep_train_only.zip` when asked,
then run all cells top to bottom. The long cell runs one atomic unit per subprocess and can
be re-run in the same runtime to resume.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='3308a0360ef8b541cfdacc7709d545a37a0efbc9'
WORK_DIR='/content/SOHO-CL'
RUN_ROOT='/content/srq_m20'
FEATURE_CACHE_DIR=RUN_ROOT+'/cifar_train_features'
OUTPUT_DIR=RUN_ROOT+'/output'
CONFIG='configs/srq_generalization_m20_adaptive_budget_criterion_train_only.json'
RUNNER='tools/srq_generalization_m20.py'
PROTOCOL='docs/research/SRQ_GENERALIZATION_M20_PROTOCOL.md'
M6_NAME='srq_generalization_m6_width_sweep_train_only.zip'
M6_SHA='b2739b9da023ebd2eedb6fdfe01c394e94f252773e847533b35350021c3d239e'
CHECKPOINT_SHA='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CHECKPOINT_SIZE=346284714
TRAIN_CACHE_SHA='ba53b82123964708123fe868a69d688bfdda4d66b1a3b7ea56329cc9ee9321dc'
TOTAL_UNITS=37
BATCH_SIZE=128
NUM_WORKERS=2
FINAL_EXPORT='/content/srq_generalization_m20_adaptive_budget_criterion_t4.zip'
assert REPO_COMMIT!='REPLACE_WITH_M20_COMMIT','Pin the commit that contains the M20 files.'

In [ ]:
# Clean pinned checkout, T4 check, and newline-normalized source lock.
import hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
def sha_raw(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1<<20),b''): h.update(block)
    return h.hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
def run_visible(command):
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1,env={**os.environ,'PYTHONUNBUFFERED':'1','PYTHONDONTWRITEBYTECODE':'1'})
    assert process.stdout is not None
    for line in process.stdout: print(line,end='')
    returncode=process.wait()
    if returncode: raise RuntimeError(f'Command failed ({returncode}): {command}')
os.chdir('/content')
if Path(WORK_DIR).exists(): shutil.rmtree(WORK_DIR)
subprocess.run(['git','clone','--no-checkout','--quiet',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--detach','--quiet',REPO_COMMIT],cwd=WORK_DIR,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=WORK_DIR,text=True).strip()==REPO_COMMIT
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Enable a Colab GPU and restart from cell 1.'
gpu_name=torch.cuda.get_device_name(0); gpu_bytes=torch.cuda.get_device_properties(0).total_memory
assert 'T4' in gpu_name,f'M20 requires Tesla T4 to stay comparable with the paper; found {gpu_name!r}.'
EXPECTED_SOURCE={
 'configs/srq_generalization_m20_adaptive_budget_criterion_train_only.json':'62e1ea13a1293d83cbfca6b7bbb6e3190048c06ac556d93b7330a62d7b5c16c5',
 'tools/srq_generalization_m20.py':'da2d86aa62733276dbb02db20a39b5fc3a7d056971eeb8e7f2836ae551363206',
 'methods/analytic_ridge/adaptive_selection.py':'e043525dd6ed26ce71f60bd28fa827d29b54c6e92d9508af53d96b2fae5a07e7',
 'tests/test_srq_generalization_m20.py':'3c5745dd958d76fe8da3168b5e0996acb4ee3a671f28a0a155cbfdc292276546',
 'tests/test_adaptive_selection.py':'843d8a857edd173e7a18cde3119237ad57bdc5e9bc90190892007fc644303f61',
 'docs/research/SRQ_GENERALIZATION_M20_PROTOCOL.md':'8c939bc874d6d2f249961cb55e68ccaaaf74edff3c4e7f621e4eeaf4084f9195',
 'methods/analytic_ridge/adaptive_upper.py':'d34eae6072ea3e8736b6fb940c7a0691c845d8217569c7b6fe33f4b9e143c5dc',
 'methods/analytic_ridge/backends.py':'cb97a6b65991e41af5f52302bcbdeac5ded6dc95774055c64c6eecdbbfb50ad3',
 'methods/analytic_ridge/qr.py':'19d24d887e60ee7fc7adb1a2265253ea261c90636aef241241e5c6faad3a507b',
 'tools/srq_generalization_m4.py':'84302805f6c71475cfcd3f7c9f148700198e96879cac6c795ef0f1bbc0f4c29e',
 'tools/srq_generalization_m5.py':'4d08e27a825fb59d300ee5909542bca4bd550a8558bb137159a176f353739a84',
 'tools/srq_generalization_m6.py':'bad119dca8b2c6e78200c81917c8e8b03a5c50923135f951d8723fd5afd2ae61',
 'tools/srq_generalization_m11.py':'b13f0ad5ed81c33a61ecca53ff536bed3f7dd8c8b0731ef4cfe5a1a8adfe0651',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda'}
for relative,expected in EXPECTED_SOURCE.items(): assert sha_source(relative)==expected,(relative,sha_source(relative),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)
print('M20 PINNED SOURCE: PASS | GPU:',gpu_name,f'{gpu_bytes/2**30:.2f} GiB')

In [ ]:
# CPU preflight: the new criterion module must reproduce the locked compressor and backend
# bitwise, and the runner must pass its tiny end-to-end pipeline before any GPU time is spent.
run_visible([sys.executable,'-B','-m','pytest','-q','-p','no:cacheprovider','tests/test_adaptive_selection.py','tests/test_srq_generalization_m20.py','tests/test_analytic_ridge_backend.py'])
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('M20 PREFLIGHT TESTS: PASS')

In [ ]:
# Upload the immutable M6 artifact; stream s2025 is identity-locked to it.
from google.colab import files
os.chdir('/content')
uploaded=files.upload()
assert set(uploaded)=={M6_NAME},f'Upload exactly {M6_NAME}; got {sorted(uploaded)}'
M6_ARTIFACT=str((Path('/content')/M6_NAME).resolve())
assert sha_raw(M6_ARTIFACT)==M6_SHA,(sha_raw(M6_ARTIFACT),M6_SHA)
os.chdir(WORK_DIR)
print('M6 ARTIFACT VERIFIED')

In [ ]:
# Locked backbone checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==CHECKPOINT_SIZE
assert sha_raw(CHECKPOINT_PATH)==CHECKPOINT_SHA
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('CHECKPOINT:',CHECKPOINT_PATH,'| CIFAR ROOT:',CIFAR_ROOT)

In [ ]:
# Materialize TRAIN features only; held-out features are forbidden.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    run_visible([sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size',str(CHECKPOINT_SIZE),'--backbone-checkpoint-sha256',CHECKPOINT_SHA,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',RUN_ROOT+'/unused','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)])
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
assert sha_raw(cache/'train.pt')==TRAIN_CACHE_SHA
print('TRAIN CACHE READY | test.pt absent')

## Long cell: 37 atomic units

Order per stream: Exact, P2B INT8, factor-MSE budgets {0, 0.05, 0.10, 0.25, 0.50, 1.00},
row-weighted budgets {0.05, 0.10, 0.25, 0.50}; one scoring-benchmark unit follows stream s2025.
Each subprocess completes exactly one unit and writes it atomically. If the runtime
disconnects, re-run cells 2, 5 and 6 (the cache check skips re-extraction only if the
runtime kept its files), then re-run this cell. Do not change budgets, streams or seeds
after seeing any unit.

In [ ]:
unit_dir=Path(OUTPUT_DIR)/'units'; unit_dir.mkdir(parents=True,exist_ok=True)
def done(): return len(list(unit_dir.glob('*.json')))
while done() < TOTAL_UNITS:
    before=done()
    print(f'M20 START/RESUME: {before}/{TOTAL_UNITS} units complete',flush=True)
    run_visible([sys.executable,'-B',RUNNER,'--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--source-m6-artifact',M6_ARTIFACT,'--output-dir',OUTPUT_DIR,'--device','cuda','--max-new-units','1','--require-clean-git'])
    after=done()
    assert after==before+1,f'Expected one new atomic unit; got {before} -> {after}'
assert Path(OUTPUT_DIR,'m20_results.json').is_file()
print(f'M20 ALL {TOTAL_UNITS} UNITS COMPLETE')

In [ ]:
# Report, then export every result even when a structural gate warns.
from google.colab import files
result_path=Path(OUTPUT_DIR)/'m20_results.json'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('GATES:',json.dumps(result['gates'],indent=2))
print('SMALLEST SUFFICIENT FACTOR-MSE BUDGET:',result['smallest_sufficient_factor_mse_budget'])
print('ROW-WEIGHTED CRITERION SUPPORTED:',result['row_weighted_criterion_supported'])
print('SCORING BENCHMARK:',json.dumps(result['scoring_benchmark'],indent=2))
for key,row in result['budget_curve'].items():
    print(f"{key:28} AIA loss {row['aia_loss_pp']['mean']:+.4f}±{row['aia_loss_pp']['sample_std']:.4f} | W err {row['final_relative_classifier_error']['mean']:.3e} | MB {row['final_total_persistent_megabytes']['mean']:.1f}")
manifest={'schema_version':1,'study_id':result['study_id'],'repo_commit':REPO_COMMIT,'config_sha256':EXPECTED_SOURCE[CONFIG],'runner_sha256':EXPECTED_SOURCE[RUNNER],'selection_module_sha256':EXPECTED_SOURCE['methods/analytic_ridge/adaptive_selection.py'],'result_sha256':sha_raw(result_path),'status':result['status']}
with zipfile.ZipFile(FINAL_EXPORT,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
    for path in sorted(Path(OUTPUT_DIR).rglob('*')):
        relative=path.relative_to(OUTPUT_DIR)
        if path.is_file() and relative.parts[0]!='cache': archive.write(path,'results/'+str(relative).replace('\\','/'))
    for relative in (CONFIG,PROTOCOL):
        archive.write(relative,'source/'+relative)
    archive.writestr('M20_ARTIFACT_MANIFEST.json',json.dumps(manifest,indent=2)+'\n')
print('FINAL EXPORT:',FINAL_EXPORT,'SHA256:',sha_raw(FINAL_EXPORT),'SIZE:',Path(FINAL_EXPORT).stat().st_size)
files.download(FINAL_EXPORT)
if result['status']!='PASS_M20_ADAPTIVE_BUDGET_CRITERION_TRAIN_ONLY': print('WARNING: preserve the artifact; do not relax gates or rerun a unit after seeing results.')